# Readability Datasets for SCE Evaluation

This notebook demonstrates the readability datasets collected for evaluating the Semantic Control Energy (SCE) readability method.

## What This Dataset Contains

The dataset includes three standardized readability datasets:

1. **CLEAR Corpus** - 4,724 examples with human readability judgments and traditional readability formula scores
2. **OneStopEnglish** - 567 examples with 3 difficulty levels (Elementary/Intermediate/Advanced)
3. **WikiLarge** - 299,062 examples of Wikipedia→Simple Wikipedia text simplification pairs

All datasets have been standardized to a common schema with:
- `input`: The text to evaluate
- `output`: Readability score or difficulty level
- Additional metadata fields (optional)

## Notebook Structure

1. **Setup**: Install dependencies and import libraries
2. **Data Loading**: Load the demo dataset from GitHub (with local fallback)
3. **Data Exploration**: Explore the structure and content of each dataset
4. **Visualization**: Visualize readability distributions and text statistics
5. **Summary**: Key findings and dataset statistics

## 1. Setup - Install Dependencies

Install required packages. On Colab, core packages (numpy, pandas, matplotlib, sklearn) are pre-installed.
On local Jupyter, we install them at Colab's exact versions to match the environment.

In [ ]:
import subprocess, sys
def _pip(*a):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
# loguru is used in the original script but not needed for demo

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

print("Setup complete!")

## 2. Imports

Import all required libraries for data loading, processing, and visualization.

In [ ]:
import json
import os
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

# For text statistics
import re

print("Imports complete!")

## 3. Data Loading

Load the demo dataset from GitHub URL with local fallback pattern.
This ensures the notebook works both in Colab (after deployment) and locally.

In [ ]:
# GitHub URL for the demo data (will work after files are pushed to GitHub)
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-210829-evaluating-embedding-based-semantic-cohe/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
        pass
    
    # Local fallback
    if os.path.exists("mini_demo_data.json"):
        print("Loading from local file...")
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

# Load the data
data = load_data()
print(f"Data loaded successfully!")
print(f"Description: {data['metadata']['description']}")
print(f"Total examples (full dataset): {data['metadata']['total_examples']}")
print(f"Sources: {', '.join(data['metadata']['sources'])}")

## 4. Data Exploration

Explore the structure and content of each dataset.
The data is organized as a list of datasets, each containing examples with standardized schema.

In [ ]:
# Print summary of each dataset
print("="*60)
print("DATASET SUMMARY")
print("="*60)

for dataset in data['datasets']:
    dataset_name = dataset['dataset']
    examples = dataset['examples']
    
    print(f"\nDataset: {dataset_name}")
    print(f"  Number of examples: {len(examples)}")
    
    # Show output value range
    outputs = [ex['output'] for ex in examples]
    print(f"  Output values: {set(outputs)}")
    
    # Show available metadata fields
    metadata_fields = [k for k in examples[0].keys() if k.startswith('metadata_')]
    print(f"  Metadata fields: {metadata_fields}")
    
    # Show first example
    print(f"\n  First example:")
    first_ex = examples[0]
    print(f"    Input (first 100 chars): {first_ex['input'][:100]}...")
    print(f"    Output: {first_ex['output']}")
    for field in metadata_fields[:3]:  # Show first 3 metadata fields
        print(f"    {field}: {first_ex.get(field, 'N/A')}")

## 5. Text Statistics

Calculate and display basic text statistics for each dataset:
- Text length (characters)
- Word count
- Sentence count

In [ ]:
def get_text_stats(text):
    """Calculate basic text statistics."""
    stats = {}
    stats['char_count'] = len(text)
    stats['word_count'] = len(text.split())
    stats['sentence_count'] = len(re.split(r'[.!?]+', text.strip())) - 1
    if stats['sentence_count'] < 1:
        stats['sentence_count'] = 1
    stats['avg_word_length'] = np.mean([len(w) for w in text.split()]) if text.split() else 0
    return stats

# Calculate statistics for all examples
all_stats = []
for dataset in data['datasets']:
    dataset_name = dataset['dataset']
    for example in dataset['examples']:
        stats = get_text_stats(example['input'])
        stats['dataset'] = dataset_name
        stats['output'] = example['output']
        all_stats.append(stats)

# Convert to DataFrame for easier analysis
df_stats = pd.DataFrame(all_stats)
print("Text Statistics Summary:")
print("="*60)
print(df_stats.groupby('dataset').agg({
    'char_count': ['mean', 'min', 'max'],
    'word_count': ['mean', 'min', 'max'],
    'sentence_count': ['mean', 'min', 'max'],
    'avg_word_length': ['mean', 'min', 'max']
}).round(2).to_string())

## 6. Visualization - Readability Distribution

Visualize the distribution of readability scores/difficulty levels across datasets.

In [ ]:
# Create subplots for each dataset
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Readability Score/Difficulty Distribution by Dataset', fontsize=14)

for idx, dataset in enumerate(data['datasets']):
    dataset_name = dataset['dataset']
    examples = dataset['examples']
    
    # Get outputs (convert to float if possible)
    outputs = []
    for ex in examples:
        try:
            outputs.append(float(ex['output']))
        except:
            outputs.append(ex['output'])
    
    ax = axes[idx]
    
    # Check if outputs are numeric
    if all(isinstance(o, float) for o in outputs):
        ax.hist(outputs, bins=10, alpha=0.7, color=cm.Set1(idx))
        ax.set_xlabel('Readability Score')
    else:
        # Categorical outputs
        counts = Counter(outputs)
        ax.bar(counts.keys(), counts.values(), color=cm.Set1(idx))
        ax.set_xlabel('Difficulty Level')
    
    ax.set_title(dataset_name.replace('_', ' ').title())
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visualization - Text Length vs Readability

Explore the relationship between text length and readability scores.

In [ ]:
# Scatter plot: text length vs readability
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Text Length vs Readability by Dataset', fontsize=14)

for idx, dataset in enumerate(data['datasets']):
    dataset_name = dataset['dataset']
    examples = dataset['examples']
    
    char_counts = [len(ex['input']) for ex in examples]
    
    # Get outputs
    outputs = []
    for ex in examples:
        try:
            outputs.append(float(ex['output']))
        except:
            outputs.append(float(ex['output']) if ex['output'].isdigit() else 0)
    
    ax = axes[idx]
    ax.scatter(char_counts, outputs, alpha=0.6, color=cm.Set1(idx))
    ax.set_xlabel('Character Count')
    ax.set_title(dataset_name.replace('_', ' ').title())
    
    # Determine y-label based on data
    if dataset_name == 'clear_corpus':
        ax.set_ylabel('Readability Score (lower = more readable)')
    else:
        ax.set_ylabel('Difficulty Level (1=Easy, 4=Hard)')
    
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Summary and Key Findings

Summarize the key characteristics of each dataset.

In [ ]:
print("="*60)
print("SUMMARY OF READABILITY DATASETS")
print("="*60)

summary_data = []

for dataset in data['datasets']:
    dataset_name = dataset['dataset']
    examples = dataset['examples']
    
    # Calculate summary statistics
    char_counts = [len(ex['input']) for ex in examples]
    word_counts = [len(ex['input'].split()) for ex in examples]
    
    outputs = []
    for ex in examples:
        try:
            outputs.append(float(ex['output']))
        except:
            outputs.append(ex['output'])
    
    summary = {
        'Dataset': dataset_name,
        'Num Examples': len(examples),
        'Avg Chars': f"{np.mean(char_counts):.0f}",
        'Avg Words': f"{np.mean(word_counts):.0f}",
        'Output Type': 'Numeric' if all(isinstance(o, float) for o in outputs) else 'Categorical'
    }
    
    if summary['Output Type'] == 'Numeric':
        summary['Output Range'] = f"{min(outputs):.2f} to {max(outputs):.2f}"
    else:
        summary['Output Range'] = f"Levels: {set(outputs)}"
    
    summary_data.append(summary)

# Display as table
df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)
print("""
1. CLEAR Corpus:
   - Contains continuous readability scores (negative values)
   - Lower scores indicate MORE readable text
   - Rich metadata including Flesch scores, grade levels

2. OneStopEnglish:
   - 3 difficulty levels (1=Elementary, 2=Intermediate, 3=Advanced)
   - Same texts rewritten at different complexity levels
   - Good for studying text simplification

3. WikiLarge:
   - Binary or multi-level difficulty (simplified vs normal Wikipedia)
   - Large-scale text simplification pairs
   - Useful for training simplification models
""")